# Download and Tile Sentinel-2 Images for 2024 over a Custom Region
This notebook demonstrates how to use the Google Earth Engine Python API to download all Sentinel-2 images for 2024 over a specified region. Because these images are large, I show how to retile these images in a separate notebook. Retiling images is helpful if you want to process these images somehow.

First off -- we just need to import the ee package, which comes from Google Earth Engine. If you are in a pip environment, downloading this is as simple as:
```bash
pip install earthengine-api
```

The authenticate line will take you to your browser, where you'll need to click through several pages. First, do not click "read only scopes" -- this will prevent you from being able to download the data (you want to be able to write data to your Google Drive). There are other methods to save the data, but for large data pulls, it's just as easy to save into your Google Drive. Beware, if you are pulling a lot of data, you might need to pay for more drive storage. It's 1-2 dollars a month, so likely worth the cost to save yourself a headache for pulling all of this data locally.

Once you authenticate, you'll then initialize the connection.

In [ ]:
import ee
ee.Authenticate(force=True)
ee.Initialize()


Successfully saved authorization token.


## Define the Region of Interest (ROI) 
First, we are going to define the criteria that we care about for this data pull. Here, we are looking at Durham, North Carolina. So, the region is defined as a bounding box: (-79.2, 35.7, -78.5, 36.4). We also only care about getting certain bands and over a certain time period, so I define that here, too.

In [24]:
# Define the region of interest
region = ee.Geometry.BBox(-79.2, 35.7, -78.5, 36.4)
start = '2023-01-01'
end   = '2023-12-31'

BANDS = ['B2','B3','B4','B8','AOT','WVP']
SCALE = 10 
max_cloud_prob = 90


criteria = ee.Filter.And(
    ee.Filter.bounds(region), ee.Filter.date(start, end)
)

Here's where things get weird -- Sentinel has products for filtering out clouds. I'm going to show how to do this, so that we only get images with clouds more or less removed. Note -- you could just load the raw images without the filter, but by joining on the cloud masks, you will get a cleaner end product.

In [25]:
sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
cp = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')

def mask_clouds(img):
    clouds = ee.Image(img.get('cloud_mask')).select('probability')
    isNotCloud = clouds.lt(max_cloud_prob)
    return img.updateMask(isNotCloud)

def mask_edges(s2_img):
    return s2_img.updateMask(
        s2_img.select('B8A').mask().updateMask(s2_img.select('B9').mask())
    )

s2Sr = sr.filter(criteria)#.map(mask_edges)
s2Clouds = cp.filter(criteria)

joined = ee.Join.saveFirst('cloud_mask').apply(s2Sr, s2Clouds, ee.Filter.equals(leftField='system:index', rightField='system:index'))
joined = ee.ImageCollection(joined).map(mask_clouds)

# Before pulling the data
I want to see how many dates there are before executing the data pull. This is what the below cell does.

In [ ]:
def set_date(img):
    return img.set('date', ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'))
with_date = joined.map(set_date)
dates = ee.List(with_date.aggregate_array('date')).distinct()
print(f" There are {len(dates.getInfo())} dates available")

 There are 144 dates available


# Pulling the data.
The cell below iterates through the list of dates, and requests a task from Google Earth Engine. For each day, we look at our list of images and select the specific date, so that we can save that image individually. We also select the bands we want.

When we export the image, we want to save a mosaic -- this is a function that effectively stitches together all of the images taken over the region of interest. Because satellites take multiple images on a day, we want to go ahead and stitch them together for the region so we get one complete image for that day. We specify a folder in which we want to save the data (associated with your Google Drive), and then I name the file logically so I can reference it in the future. 

In [ ]:
for d in dates.getInfo():
    day = with_date.filter(ee.Filter.eq('date', d)).select(BANDS)

    # date_str = day.get('date').getInfo()  # safe: one call per task creation
    task = ee.batch.Export.image.toDrive(
        image = day.mosaic(),
        description = f'S2_daily_{d}_masked_B2_B3_B4_B8_AOT_WVP',
        folder = 'GEE_2023',
        fileNamePrefix = f'S2_daily_{d}_masked',
        region = region,
        scale = SCALE,
        maxPixels = 1e13
    )
    task.start()

That's it -- all of the tasks have been executed, with a single task for each file we want to pull. This will likely take several hours, as the tasks make their way through the GEE queue. We can look at the status of our tasks using the cell below. The tasks are indexed in reverse order -- that is, our first task will have our last index.

In [ ]:
# List and print the status of all Earth Engine export tasks
tasks = ee.batch.Task.list()

task_idx = -1 # check our first task
task = tasks[task_idx]
status = task.status()
print('Description:', task.config.get('description', 'N/A'))
print('State:', status['state'])
if 'error_message' in status:
    print('Error message:', status['error_message'])
else:
    print('No error message.')

Description: S2_daily_2023-01-03_masked_B2_B3_B4_B8_AOT_WVP
State: COMPLETED
No error message.


In [ ]:
# Or, just print out the list of tasks.
tasks

[<Task 3OLU7PHGNJVKGQ22VFPO5MBJ EXPORT_IMAGE: S2_daily_2023-12-29_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task IBMRLSM7WU6JLNRRKKWC5X72 EXPORT_IMAGE: S2_daily_2023-12-26_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task KQ34RZN65PDBCNSXLBNCQKBL EXPORT_IMAGE: S2_daily_2023-12-24_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task BEBVUWROMCKHIGBURZ75YTDQ EXPORT_IMAGE: S2_daily_2023-12-21_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task KDTCZJ6BYYU3PHZSNNZAM5OE EXPORT_IMAGE: S2_daily_2023-12-19_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task FPEFACWY4UOR7WDNQXENR5KQ EXPORT_IMAGE: S2_daily_2023-12-16_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task JQ3G52ITBDMFR5CXHT4J5IGS EXPORT_IMAGE: S2_daily_2023-12-14_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task JRPALWKLEYRXUECWQAOI4ILT EXPORT_IMAGE: S2_daily_2023-12-11_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task N4WRQZMKGPIKDJAHJ6PZDZDI EXPORT_IMAGE: S2_daily_2023-12-09_masked_B2_B3_B4_B8_AOT_WVP (READY)>,
 <Task 2DEUTRR4S5WHTQJBVSBPDMOY EXPORT_IMAGE: S2_daily_2023-12-06_masked_

---
**Note:** Exporting all tiles for all images may result in a very large number of tasks. Consider filtering by fewer dates or a smaller region for testing.